# LangChain Agent Benchmark 01: Models, Prompts, Context, and Outputs

This notebook benchmarks model and prompt variables while keeping the task constant. It uses LangChain prompt templates together with Hugging Face chat models and the `.invoke()` and `.stream()` interfaces directly.


In [ ]:
# Run once per environment. Keep optional provider packages commented until needed.
%pip install -qU langchain langchain-core langchain-community langchain-huggingface langchain-text-splitters langgraph pandas pydantic
# Optional local/vector-store packages:
# %pip install -qU faiss-cpu langchain-chroma langchain-qdrant qdrant-client sentence-transformers langchain-huggingface


In [ ]:
import os
import time
from typing import Literal

import pandas as pd
from huggingface_hub import InferenceClient

pd.set_option("display.max_colwidth", None)
pd.set_option("display.width", None)

HF_TOKEN = os.getenv("HUGGINGFACEHUB_API_TOKEN") or os.getenv("HF_TOKEN")
HF_TIMEOUT = 120
GEMMA_MODEL = "google/gemma-3-12b-it"
GEMMA_PROVIDER = "featherless-ai"
MODEL_SPECS = [
    {"model": "google/gemma-3-12b-it", "provider": "featherless-ai"},
    # {"model": "meta-llama/Llama-3.1-8B-Instruct", "provider": "auto"},
    # {"model": "Qwen/Qwen2.5-7B-Instruct", "provider": "together"},
    # {"model": "m42-health/Llama3-Med42-8B", "provider": "featherless-ai"},
    # {"model": "Qwen/Qwen2.5-0.5B-Instruct", "provider": "featherless-ai"},
]

# if not HF_TOKEN:
#     print("Set HUGGINGFACEHUB_API_TOKEN or HF_TOKEN before running the examples.")


## 1. LLM choice

LLM choice affects factual accuracy, biological knowledge, reasoning quality, hallucination behavior, speed, verbosity, and code quality. This is because it relies completely on the content the model has seen during training, somewhat imitating it.


In [ ]:
questions = [
    "Explain why batch correction matters in single-cell RNA-seq analysis.",
    # "Explain the role of STAT3 in cancer.",
]
rows = []

for question in questions:
    for spec in MODEL_SPECS:
        start = time.perf_counter()
        answer = None
        error = None
        try:
            client_kwargs = {
                "model": spec["model"],
                "api_key": HF_TOKEN,
                "timeout": HF_TIMEOUT,
            }
            if spec["provider"] != "auto":
                client_kwargs["provider"] = spec["provider"]
            client = InferenceClient(**client_kwargs)

            for attempt in range(3):
                try:
                    response = client.chat_completion(
                        messages=[{"role": "user", "content": question}],
                        max_tokens=120,
                        temperature=0,
                    )
                    answer = response.choices[0].message.content
                    break
                except Exception as exc:
                    if attempt == 2:
                        raise
                    time.sleep(2)
        except Exception as exc:
            error = repr(exc)

        rows.append(
            {
                "question": question,
                "model": spec["model"],
                "provider": spec["provider"],
                "seconds": round(time.perf_counter() - start, 3),
                "answer": answer,
                "error": error,
            }
        )

df = pd.DataFrame(rows)
df["top_context"] = df["answer"]
df.style.set_properties(
    subset=["top_context"],
    **{"white-space": "pre-wrap", "text-align": "left", "min-width": "650px"},
)


## 2. Temperature

Temperature controls sampling randomness: lower values produce more deterministic answers while higher values increase creativity (can also increase hallucination risk). This works by flattening the output probabilities of the model.

In [ ]:
question = "Suggest three biologically plausible explanations for a rare cell cluster with high ribosomal gene expression."
rows = []

for temperature in [0, 0.3, 0.8]:
    client = InferenceClient(model=GEMMA_MODEL, provider=GEMMA_PROVIDER, api_key=HF_TOKEN, timeout=HF_TIMEOUT)
    start = time.perf_counter()
    response = client.chat_completion(
        messages=[{"role": "user", "content": question}],
        max_tokens=120,
        temperature=temperature,
    )
    rows.append({"temperature": temperature, "seconds": round(time.perf_counter() - start, 3), "answer": response.choices[0].message.content})

df = pd.DataFrame(rows)
df["top_context"] = df["answer"]
df.style.set_properties(
    subset=["top_context"],
    **{"white-space": "pre-wrap", "text-align": "left", "min-width": "650px"},
)


## 3. System prompt

The system prompt sets high-level behavior such as persona, assumption, criteria for tool usage, and any instruction that should always be followed. This is injected at the first turn of the conversation and is kept in memory across all turns. Implemented via `SystemMessage` inside a `ChatPromptTemplate`. 

In [ ]:
question = "A sample has high mitochondrial RNA and low detected genes. Is it a dying-cell population?"
rows = []
client = InferenceClient(model=GEMMA_MODEL, provider=GEMMA_PROVIDER, api_key=HF_TOKEN, timeout=HF_TIMEOUT)

messages = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": question},
]
response = client.chat_completion(messages=messages, max_tokens=120, temperature=0)
rows.append({"system_prompt": "minimal", "answer": response.choices[0].message.content})

messages = [
    {"role": "system", "content": "You are a molecular biologist who explains concepts clearly to computational biology students."},
    {"role": "user", "content": question},
]
response = client.chat_completion(messages=messages, max_tokens=120, temperature=0)
rows.append({"system_prompt": "biologist persona", "answer": response.choices[0].message.content})

messages = [
    {"role": "system", "content": "You are a strict scientific assistant. Separate evidence from speculation, state uncertainty, and avoid unsupported claims."},
    {"role": "user", "content": question},
]
response = client.chat_completion(messages=messages, max_tokens=120, temperature=0)
rows.append({"system_prompt": "strict scientific", "answer": response.choices[0].message.content})

df = pd.DataFrame(rows)
df["top_context"] = df["answer"]
df.style.set_properties(
    subset=["top_context"],
    **{"white-space": "pre-wrap", "text-align": "left", "min-width": "650px"},
)


## 4. Prompt template

This is a scaffold that controls how the task is framed and allows the injection of dynamic data like user inputs or variables at runtime. Implemented via `PromptTemplate` or `ChatPromptTemplate`. 

In [ ]:
observation = "a T-cell cluster has high interferon-stimulated genes after stimulation"
rows = []
client = InferenceClient(model=GEMMA_MODEL, provider=GEMMA_PROVIDER, api_key=HF_TOKEN, timeout=HF_TIMEOUT)

prompt = f"Explain whether {observation} is biologically meaningful."
response = client.chat_completion(messages=[{"role": "user", "content": prompt}], max_tokens=120, temperature=0)
rows.append({"template": "free form", "answer": response.choices[0].message.content})

prompt = (
    "Example:\n"
    "Observation: high MALAT1 in low-quality nuclei.\n"
    "Answer: This may reflect nuclear RNA content or technical quality; validate with QC metrics and markers.\n\n"
    f"Observation: {observation}\n"
    "Answer:"
)
response = client.chat_completion(messages=[{"role": "user", "content": prompt}], max_tokens=120, temperature=0)
rows.append({"template": "few shot", "answer": response.choices[0].message.content})

prompt = (
    f"Observation: {observation}\n"
    "Return sections: Interpretation | Alternative explanations | Checks | Confidence."
)
response = client.chat_completion(messages=[{"role": "user", "content": prompt}], max_tokens=120, temperature=0)
rows.append({"template": "structured", "answer": response.choices[0].message.content})

df = pd.DataFrame(rows)
df["top_context"] = df["answer"]
df.style.set_properties(
    subset=["top_context"],
    **{"white-space": "pre-wrap", "text-align": "left", "min-width": "650px"},
)


## 5. Context length

Context length controls how much background information the model receives, which can improve grounding but can also introduce distractions, truncation, and irrelevant details.


In [ ]:
base_context = "Protocol note: Samples were PBMCs stimulated with IFN-beta for 6 hours. Mitochondrial reads above 20% were filtered."
small_context = base_context
large_context = "\n".join([base_context] + [
    "Marker note: IFIT1, ISG15, MX1, and OAS1 indicate interferon response.",
    "QC note: doublet scores above 0.25 were removed.",
    "Batch note: donor and library chemistry can confound differential expression.",
] * 8)
too_much_context = large_context + "\n" + "\n".join(
    [f"Irrelevant lab inventory line {i}: freezer box metadata unrelated to expression." for i in range(120)]
)

contexts = {"no_context": "", "small_context": small_context, "large_context": large_context, "too_much_context": too_much_context}
question = "Why might IFIT1 and ISG15 be elevated, and what caveats should be checked?"
rows = []
for label, ctx in contexts.items():
    client = InferenceClient(model=GEMMA_MODEL, provider=GEMMA_PROVIDER, api_key=HF_TOKEN, timeout=HF_TIMEOUT)
    prompt = f"Context:\n{ctx}\n\nQuestion: {question}" if ctx else question
    start = time.perf_counter()
    response = client.chat_completion(messages=[{"role": "user", "content": prompt}], max_tokens=120, temperature=0)
    rows.append({"context": label, "seconds": round(time.perf_counter() - start, 3), "answer": response.choices[0].message.content})

df = pd.DataFrame(rows)
df["top_context"] = df["answer"]
df.style.set_properties(
    subset=["top_context"],
    **{"white-space": "pre-wrap", "text-align": "left", "min-width": "650px"},
)


## 6. Output parsing

This determines in what format the LLM produces the output, eg. plain text, JSON, pydantic. This is relevant in the context of agents because downstream processes that use LLM's output as input may require it in specific formats. Additionally, structured formats enforce strict validation checking.  

In [ ]:
QUESTION = "Interpret high IFIT1 and ISG15 expression in treated PBMCs."
CONTEXT = "Treated PBMC samples were stimulated with interferon beta for 6 hours. IFIT1 and ISG15 are interferon-stimulated genes."


### i. Free text

Free text output is easy for humans to read but unreliable for downstream parsing and automated evaluation.


In [ ]:
client = InferenceClient(model=GEMMA_MODEL, provider=GEMMA_PROVIDER, api_key=HF_TOKEN, timeout=HF_TIMEOUT)
response = client.chat_completion(messages=[{"role": "user", "content": QUESTION}], max_tokens=120, temperature=0)
print(response.choices[0].message.content)


### ii. JSON output

JSON output asks the model to return machine-readable fields, but malformed JSON can still occur without validation or native structured output.


In [ ]:
from langchain_core.output_parsers import JsonOutputParser, PydanticOutputParser, StrOutputParser

parser = JsonOutputParser()
prompt = f"Return valid JSON only with keys answer, confidence, caveats.\n\nQuestion: {QUESTION}"
client = InferenceClient(model=GEMMA_MODEL, provider=GEMMA_PROVIDER, api_key=HF_TOKEN, timeout=HF_TIMEOUT)
response = client.chat_completion(messages=[{"role": "user", "content": prompt}], max_tokens=120, temperature=0)
parser.parse(response.choices[0].message.content)


### iii. Pydantic parser

Pydantic parser validates types and required fields, making outputs more reliable for downstream code.


In [ ]:
from pydantic import BaseModel, Field, ValidationError

class BioAnswer(BaseModel):
    answer: str = Field(description="Concise answer")
    confidence: Literal["low", "medium", "high"]
    caveats: list[str] = Field(default_factory=list)

parser = PydanticOutputParser(pydantic_object=BioAnswer)
prompt = f"Answer the question.\n{parser.get_format_instructions()}\n\nQuestion: {QUESTION}"
client = InferenceClient(model=GEMMA_MODEL, provider=GEMMA_PROVIDER, api_key=HF_TOKEN, timeout=HF_TIMEOUT)
response = client.chat_completion(messages=[{"role": "user", "content": prompt}], max_tokens=120, temperature=0)
parser.parse(response.choices[0].message.content)


### iv. Structured output API

Structured output API uses provider-native or tool-call based schema enforcement where supported, reducing manual parsing failures.


In [ ]:
parser = PydanticOutputParser(pydantic_object=BioAnswer)
prompt = (
    "Return output that matches the requested schema exactly.\n\n"
    f"Question: {QUESTION}\n\n"
    f"{parser.get_format_instructions()}"
)
client = InferenceClient(model=GEMMA_MODEL, provider=GEMMA_PROVIDER, api_key=HF_TOKEN, timeout=HF_TIMEOUT)
response = client.chat_completion(messages=[{"role": "user", "content": prompt}], max_tokens=120, temperature=0)
parser.parse(response.choices[0].message.content)


### v. Markdown formatting

Markdown formatting controls the shape of the answer, such as plain text, tables, or bullet lists, which affects readability and downstream copy-paste usability.


In [ ]:
formats = {
    "plain_text": "Answer in one short paragraph.",
    "table": "Answer as a markdown table with columns Finding, Interpretation, Caveat.",
    "bullet_list": "Answer as concise bullet points.",
}
question = "Summarize how to interpret marker genes, QC metrics, and batch effects in scRNA-seq."
rows = []
for label, instruction in formats.items():
    client = InferenceClient(model=GEMMA_MODEL, provider=GEMMA_PROVIDER, api_key=HF_TOKEN, timeout=HF_TIMEOUT)
    prompt = f"{instruction}\n\n{question}"
    response = client.chat_completion(messages=[{"role": "user", "content": prompt}], max_tokens=120, temperature=0)
    rows.append({"format": label, "answer": response.choices[0].message.content})

for row in rows:
    print("\n###", row["format"], "\n", row["answer"])


## 7. Retry strategies

Retry strategies make model workflows more robust when a provider call fails, the model returns unparsable output, or a backup model is needed.


### i. Retry parser

A parser retry asks the model to repair malformed structured output after a parsing or validation failure.


In [ ]:
parser = PydanticOutputParser(pydantic_object=BioAnswer)
client = InferenceClient(model=GEMMA_MODEL, provider=GEMMA_PROVIDER, api_key=HF_TOKEN, timeout=HF_TIMEOUT)
instructions = parser.get_format_instructions()

response = client.chat_completion(messages=[{"role": "user", "content": f"{QUESTION}\n{instructions}"}], max_tokens=120, temperature=0)
raw = response.choices[0].message.content

try:
    parsed = parser.parse(raw)
    attempt = 1
except Exception as exc:
    repair_prompt = f"{QUESTION}\n{instructions}\nPrevious parse error: {exc!r}\nReturn corrected output only."
    response = client.chat_completion(messages=[{"role": "user", "content": repair_prompt}], max_tokens=120, temperature=0)
    raw = response.choices[0].message.content
    parsed = parser.parse(raw)
    attempt = 2

parsed, attempt, raw


### ii. No retry policy

Without a retry policy, transient provider errors, rate limits, and network interruptions surface immediately.


In [ ]:
client = InferenceClient(model=GEMMA_MODEL, provider=GEMMA_PROVIDER, api_key=HF_TOKEN, timeout=HF_TIMEOUT)
response = client.chat_completion(messages=[{"role": "user", "content": QUESTION}], max_tokens=120, temperature=0)
response.choices[0].message.content


### iii. Retry policy

A retry policy automatically re-executes failed calls, improving robustness at the cost of extra latency and token use.


In [ ]:
client = InferenceClient(model=GEMMA_MODEL, provider=GEMMA_PROVIDER, api_key=HF_TOKEN, timeout=HF_TIMEOUT)
last_exc = None
for attempt in range(3):
    try:
        response = client.chat_completion(messages=[{"role": "user", "content": QUESTION}], max_tokens=120, temperature=0)
        break
    except Exception as exc:
        last_exc = exc
        if attempt == 2:
            raise
        time.sleep(2)
response.choices[0].message.content


### iv. Fallback model

A fallback model tries another model when the first model fails. In production code, you would usually stop after the first successful response.


In [ ]:
rows = []

for spec in [
    {"model": "google/gemma-3-12b-it", "provider": "featherless-ai"},
    {"model": "google/gemma-3-27b-it", "provider": "featherless-ai"},
]:
    try:
        client = InferenceClient(model=spec["model"], provider=spec["provider"], api_key=HF_TOKEN, timeout=HF_TIMEOUT)
        response = client.chat_completion(messages=[{"role": "user", "content": QUESTION}], max_tokens=120, temperature=0)
        rows.append({"model": spec["model"], "answer": response.choices[0].message.content, "error": None})
    except Exception as exc:
        rows.append({"model": spec["model"], "answer": None, "error": repr(exc)})

df = pd.DataFrame(rows)
df["top_context"] = df["answer"]
df.style.set_properties(
    subset=["top_context"],
    **{"white-space": "pre-wrap", "text-align": "left", "min-width": "650px"},
)


## 8. Streaming

Streaming controls whether tokens are delivered incrementally, which affects user experience and perceived latency without necessarily changing the final answer.


In [ ]:
client = InferenceClient(model=GEMMA_MODEL, provider=GEMMA_PROVIDER, api_key=HF_TOKEN, timeout=HF_TIMEOUT)
prompt = "Give a concise explanation of pseudobulk differential expression."

chunks = []
start = time.perf_counter()
for chunk in client.chat_completion(messages=[{"role": "user", "content": prompt}], max_tokens=120, temperature=0, stream=True):
    if chunk.choices and chunk.choices[0].delta.content:
        text = chunk.choices[0].delta.content
        chunks.append(text)
        print(text, end="")
print("\n\nstream_seconds:", round(time.perf_counter() - start, 3))


## 9. Citation generation

Citation generation asks the model to attach source references to claims, which improves auditability only when the citations are grounded in provided sources or retrieval metadata.


In [ ]:
context = (
    "[S1] IFIT1, IFIT3, ISG15, MX1, and OAS genes are common interferon-stimulated genes.\n"
    "[S2] High mitochondrial RNA fractions can indicate low-quality or stressed cells, but thresholds are tissue and protocol dependent."
)
prompt = f"Use only the sources below and cite each claim with [S1] or [S2].\n\n{context}\n\nQuestion: Interpret high IFIT1 and high mitochondrial RNA."

client = InferenceClient(model=GEMMA_MODEL, provider=GEMMA_PROVIDER, api_key=HF_TOKEN, timeout=HF_TIMEOUT)
response = client.chat_completion(messages=[{"role": "user", "content": prompt}], max_tokens=120, temperature=0)
print(response.choices[0].message.content)
